# Лабораторная работа 4_2

## Т-тесты. Гендерные различия психологических показателей

**ФИО:** Аюпова В.Р.
**Группа:** ЕТ-212
**Дата:** 21.10.2026

## Условие

1. Загрузить `Психология.xlsx` в DataFrame в Pandas.
2. Оценить, для каких показателей существуют гендерные различия, а для каких нет:
   a. сформулировать гипотезы
   b. правильно выбрать Т-тест и правильно задать параметры
   c. для каждого показателя — статистика + p-значение + вывод
3. Где гендерные различия проявились сильнее всего, а где слабее?

## Часть 0. Импорт библиотек

In [2]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings

# отключаем лишние предупреждения для чистоты вывода
warnings.filterwarnings("ignore")

## Пункт 1. Загрузка данных

**Что делаем:** читаем файл, смотрим структуру и типы.

In [3]:
df_psy = pd.read_excel("data/Психологи.xlsx")

print("Размер датасета:", df_psy.shape)
print("\nСтолбцы:")
print(df_psy.columns.tolist())
print("\nПервые 5 строк:")
df_psy.head()

Размер датасета: (46, 19)

Столбцы:
['учащиеся', ' пол', ' класс', ' профиль ВУЗа', ' осведомленность', ' скрытые фигуры', ' пропущенные слова', ' арифметика', ' понятливость', ' исключение  изображений', ' аналогии', ' числовые ряды', ' умозаключения', ' геометрическое  сложение', ' заучивание слов', ' средний IQ', ' экстраверсия- интроверсия', ' нейротизм', ' средняя отметка']

Первые 5 строк:


,учащиеся,пол,класс,профиль ВУЗа,осведомленность,скрытые фигуры,пропущенные слова,арифметика,понятливость,исключение изображений,аналогии,числовые ряды,умозаключения,геометрическое сложение,заучивание слов,средний IQ,экстраверсия- интроверсия,нейротизм,средняя отметка
0,1,ж,гум,0,12,9,11,8,8,11,13,8,12,10,11,103,15,7,3.93
1,2,ж,гум,1,10,12,12,11,10,12,9,12,8,11,11,107,13,17,4.27
2,3,м,мат,1,11,8,9,11,11,12,9,11,9,11,8,100,10,19,3.87
3,4,ж,гум,0,14,12,14,13,13,9,9,9,12,11,12,116,13,11,4.57
4,5,ж,гум,0,12,12,9,10,9,12,5,10,3,11,9,93,16,8,4.14


**Подготовка:** убираем лишние пробелы в названиях столбцов (в исходном Excel они есть в начале каждого имени), чтобы к ним было удобно обращаться в коде.

In [4]:
# Убираем пробелы в начале и конце названий столбцов
df_psy.columns = df_psy.columns.str.strip()

print("Очищенные названия столбцов:")
print(df_psy.columns.tolist())

Очищенные названия столбцов:
['учащиеся', 'пол', 'класс', 'профиль ВУЗа', 'осведомленность', 'скрытые фигуры', 'пропущенные слова', 'арифметика', 'понятливость', 'исключение  изображений', 'аналогии', 'числовые ряды', 'умозаключения', 'геометрическое  сложение', 'заучивание слов', 'средний IQ', 'экстраверсия- интроверсия', 'нейротизм', 'средняя отметка']


## Пункт 2. Оценка гендерных различий

**Гипотезы для каждого показателя:**
- **H0:** средние значения показателя у юношей и девушек равны (различий нет).
- **H1:** средние значения показателя у юношей и девушек различаются (различия есть).

**Выбор теста:**
- Выборки независимые (разные учащиеся) → `ttest_ind`.
- Предварительно проверяем равенство дисперсий критерием Левена:
  - если p > 0.05 → `equal_var=True` (Стьюдент);
  - если p ≤ 0.05 → `equal_var=False` (Уэлч).

In [5]:
def test_gender_diff(df, col, alpha=0.05):
    # Берем данные по юношам и девушкам (в датасете 'м' и 'ж')
    males = df[df["пол"] == "м"][col].dropna()
    females = df[df["пол"] == "ж"][col].dropna()

    # Проверка равенства дисперсий (Левен)
    lev_stat, lev_p = stats.levene(males, females)
    equal_var = lev_p > alpha

    # Сам t-тест
    t_stat, p_val = stats.ttest_ind(males, females, equal_var=equal_var)

    # Вывод
    conclusion = "ЕСТЬ различия" if p_val < alpha else "НЕТ различий"
    test_name = "Стьюдент" if equal_var else "Уэлч"

    print(f"{col:30} | Сред. М: {males.mean():5.1f} | Сред. Ж: {females.mean():5.1f} | "
          f"Левен p: {lev_p:.3f} | t-тест: {test_name:7} | t = {t_stat:6.2f} | p = {p_val:.4f} | {conclusion}")

**Что делаем:** применяем тест ко всем числовым психологическим показателям (начиная с 5-го столбца и до конца, исключая служебные).

In [6]:
# Берем только числовые показатели, пропуская первые 4 (учащиеся, пол, класс, профиль)
# и исключая последний, если он не нужен (но по заданию "для каждого показателя" - берем все числовые)
numeric_cols = df_psy.columns[4:] 

print(f"{'Показатель':30} | {'Сред М':7} | {'Сред Ж':7} | {'Левен p':7} | {'t-тест':7} | {'t-stat':6} | {'p-value':6} | Вывод")
print("-" * 120)

for col in numeric_cols:
    test_gender_diff(df_psy, col)

Показатель                     | Сред М  | Сред Ж  | Левен p | t-тест  | t-stat | p-value | Вывод
------------------------------------------------------------------------------------------------------------------------
осведомленность                | Сред. М:  10.3 | Сред. Ж:  11.3 | Левен p: 0.402 | t-тест: Стьюдент | t =  -1.47 | p = 0.1478 | НЕТ различий
скрытые фигуры                 | Сред. М:   9.0 | Сред. Ж:  10.6 | Левен p: 0.618 | t-тест: Стьюдент | t =  -2.01 | p = 0.0504 | НЕТ различий
пропущенные слова              | Сред. М:   9.9 | Сред. Ж:  10.4 | Левен p: 0.857 | t-тест: Стьюдент | t =  -0.62 | p = 0.5373 | НЕТ различий
арифметика                     | Сред. М:  11.7 | Сред. Ж:  10.7 | Левен p: 0.527 | t-тест: Стьюдент | t =   1.67 | p = 0.1023 | НЕТ различий
понятливость                   | Сред. М:   9.9 | Сред. Ж:  10.2 | Левен p: 0.889 | t-тест: Стьюдент | t =  -0.53 | p = 0.6019 | НЕТ различий
исключение  изображений        | Сред. М:   9.3 | Сред. Ж:  10.3 | Леве

**Вывод по п.2:**  
Для всех числовых показателей были сформулированы гипотезы: 
- H0: средние значения у юношей и девушек равны;
- H1: средние значения различаются.

По критерию Левена для всех показателей p > 0.05, следовательно, дисперсии в группах равны, и для всех тестов использовался классический t-тест Стьюдента (`equal_var=True`).

По результатам t-теста при α = 0.05 статистически значимые гендерные различия обнаружены только по показателю **«экстраверсия-интроверсия»** (t = -3.44, p = 0.0013). У девушек средний балл по этой шкале (14.6) выше, чем у юношей (11.6). По остальным 14 показателям (включая IQ, успеваемость, нейротизм и др.) p-value > 0.05, поэтому нулевая гипотеза не отвергается — значимых различий между юношами и девушками нет.

Отдельно можно отметить показатель **«скрытые фигуры»** (p = 0.0504), который находится на границе уровня значимости, но формально не проходит порог.

## Пункт 3. Где различия проявились сильнее, а где слабее?

**Что делаем:** сортируем показатели по возрастанию p-value, чтобы увидеть, где различия наиболее статистически значимы (маленький p — сильные различия), а где их нет совсем (p близко к 1).

In [7]:
# Собираем все p-value в список для наглядной сортировки
results = []
for col in df_psy.columns[4:]:
    males = df_psy[df_psy["пол"] == "м"][col].dropna()
    females = df_psy[df_psy["пол"] == "ж"][col].dropna()
    lev_stat, lev_p = stats.levene(males, females)
    t_stat, p_val = stats.ttest_ind(males, females, equal_var=(lev_p > 0.05))
    results.append({'Показатель': col, 'p-value': p_val, 't-stat': t_stat})

res_df = pd.DataFrame(results).sort_values('p-value').reset_index(drop=True)
print("Показатели, отсортированные по возрастанию p-value:")
res_df.round(4)

Показатели, отсортированные по возрастанию p-value:


,Показатель,p-value,t-stat
0,экстраверсия- интроверсия,0.0013,-3.4441
1,скрытые фигуры,0.0504,-2.0113
2,арифметика,0.1023,1.6686
3,нейротизм,0.1136,1.6145
4,числовые ряды,0.1163,1.6023
5,заучивание слов,0.1318,-1.5355
6,средняя отметка,0.1434,-1.4897
7,осведомленность,0.1478,-1.4732
8,исключение изображений,0.1816,-1.3573
9,геометрическое сложение,0.1880,1.3373


**Вывод по п.3:**  
Самые сильные (статистически значимые) различия проявились по шкале **«экстраверсия-интроверсия»** (минимальное p-value = 0.0013). Девушки более экстравертированы, чем юноши.

Самые слабые различия (практически полное отсутствие) — по показателю **«средний IQ»** (p = 0.8691), а также по **«умозаключения»** (p = 0.7639) и **«понятливость»** (p = 0.6019). Здесь средние значения у юношей и девушек почти идентичны.

## Общий вывод

В ходе лабораторной работы были выполнены следующие задачи:

1. **Загрузка данных.** Датасет «Психологи» успешно загружен в DataFrame (46 строк, 19 столбцов). Названия столбцов были очищены от лишних пробелов для корректной работы кода.
2. **Формулировка гипотез и выбор теста.** Для каждого числового показателя выдвинуты гипотезы H0 (средние равны) и H1 (средние различаются). Поскольку выборки независимые, использован `ttest_ind`. Перед каждым тестом проводилась проверка равенства дисперсий (критерий Левена) — для всех показателей дисперсии оказались равны, поэтому применялся t-тест Стьюдента.
3. **Результаты.** Из 15 числовых показателей статистически значимые гендерные различия обнаружены только по шкале **«экстраверсия-интроверсия»** (p = 0.0013). Девушки в среднем более экстравертированы. По остальным показателям (IQ, отдельные когнитивные тесты, нейротизм, средняя отметка) значимых различий не выявлено.
4. **Сила различий.** Самые сильные различия — по экстраверсии. Самые слабые — по общему IQ и умозаключениям.

**Инструменты:** Python 3, Pandas, NumPy, SciPy.